In [ ]:
# This cell is added by Sphinx-Gallery
# It can be customized to whatever you like


# Read Lean Proofs with LeanReader: Tracing Every Line of a Formal Development

.. currentmodule:: scikitplot.corpus

[Lean](https://lean-lang.org/) proofs can run to millions of lines, far past
what a person can keep track of. :class:`LeanReader` splits a ``.lean`` file
into one chunk per Lean command: a ``theorem``, a ``def``, a ``namespace``.
Each chunk records where it is, what it declares and the audit flags it
raises. This example goes from the simplest case to the hard ones:

1. one file: what a chunk carries;
2. statements without proofs (``proofs="drop"``), for reading claims;
3. audit flags: ``sorry``, ``native_decide``, ``axiom``, code after ``#exit``;
4. choosing what to read: ``kinds``, ``watch``, ``commands``;
5. very long proofs read in parts (``lines_per_chunk``);
6. Lean 3 sources (``dialect="lean3"``);
7. hostile input: unclosed comments, too-deep scopes;
8. a generated project of thousands of declarations, with a check that
   every line lands in exactly one chunk.

Everything runs on files the example writes: no Lean toolchain, no download.

<div class="alert alert-info"><h4>Note</h4>

The reader is lexical: it reads the text, it does not check proofs.
A file without flags is not proved correct; that is for Lean's kernel
(``lake build``, ``#print axioms``).

</div>


In [ ]:
# Authors: The scikit-plots developers
# SPDX-License-Identifier: BSD-3-Clause

In [ ]:
from __future__ import annotations

import tempfile
from collections import Counter
from pathlib import Path

from scikitplot.corpus import LEAN_FLAGS, LeanReader

work = Path(tempfile.mkdtemp(prefix="lean_reader_"))


def read(text, name="Demo.lean", **options):
    """Write *text* to a file and return the chunks LeanReader makes of it."""
    path = work / name
    path.write_text(text, encoding="utf-8")
    return list(LeanReader(input_path=path, **options).get_raw_chunks())


def show(chunks, *keys):
    for c in chunks:
        extra = {k: c[k] for k in keys if k in c}
        print(f"{c['line_start']:>3}-{c['line_end']:<3} {c['lean_kind']:<10}",
              c.get("declaration", ""), extra)

## 1. One file
Imports, namespaces, doc comments and declarations each become a chunk.
A doc comment stays with the declaration it documents; ``declaration`` is
the full name, namespace included.



In [ ]:
source = """\
import Mathlib.Data.Nat.Basic

namespace Demo

/-- Adding zero changes nothing. -/
theorem add_zero' (n : Nat) : n + 0 = n := by
  simp

def double (n : Nat) : Nat := 2 * n

end Demo
"""
show(read(source), "namespace", "imports")

## 2. Statements without proofs
``proofs="drop"`` keeps what a declaration claims and leaves out how it is
proved. ``proof_line_start``/``proof_line_end`` say where the proof was.



In [ ]:
for c in read(source, proofs="drop"):
    if c["lean_kind"] == "theorem":
        print(repr(c["text"]), c.get("proof_line_start"), c.get("proof_line_end"))

## 3. Audit flags
Some constructs let a "proof" through without the kernel checking it.
The reader flags them in code only, never in comments or strings. A
comment line belongs to the declaration after it (``fast`` starts at 2).
Everything after ``#exit`` is flagged too, since Lean never reads it.



In [ ]:
print(sorted(set(LEAN_FLAGS.values())))
risky = """\
theorem gap : 1 = 2 := by sorry
-- a comment saying sorry is not flagged
theorem fast : 2 ^ 10 = 1024 := by native_decide
axiom trust_me : False
#exit
theorem never_checked : True := trivial
"""
show(read(risky, "Risky.lean"), "flags")

## 4. Choosing what to read
``kinds`` keeps some kinds of command; ``watch`` marks chunks naming the
identifiers you care about; ``commands`` adds a project's own command
keywords, so their uses become chunks of their own.



In [ ]:
print([c["declaration"] for c in read(source, kinds=["theorem"])])
print([c.get("watched") for c in read(source, watch=["double"]) if "watched" in c])
custom = "theorem a : True := trivial\nmy_check a\ntheorem b : True := trivial\n"
print([c["lean_kind"] for c in read(custom, "Custom.lean")])
print([c["lean_kind"] for c in read(custom, "Custom.lean", commands=["my_check"])])

## 5. Very long proofs
A proof of thousands of lines is read in parts of ``lines_per_chunk``
lines; ``part`` numbers them, and every part keeps the declaration name.



In [ ]:
long_proof = "theorem big : True := by\n" + "  skip\n" * 1_000 + "  trivial\n"
parts = read(long_proof, "Long.lean", lines_per_chunk=300)
print([(c["part"], c["line_start"], c["line_end"]) for c in parts])

## 6. Lean 3
Lean 3 proofs use ``begin ... end`` blocks; ``dialect="lean3"`` reads them
as one command each.



In [ ]:
lean3 = """\
import data.nat.basic

theorem add_comm' (a b : ℕ) : a + b = b + a :=
begin
  exact nat.add_comm a b,
end
"""
show(read(lean3, "Old.lean", dialect="lean3"))

## 7. Hostile input
A comment that never closes swallows the rest of the file as comment, as
Lean would. Scopes nested deeper than ``max_scope_depth`` are refused with
a clear error instead of using unbounded memory.



In [ ]:
print([c["section_type"] for c in read("/- never closed\ntheorem x : True := trivial\n", "Open.lean")])
try:
    read("namespace N\n" * 1_000, "Deep.lean", max_scope_depth=256)
except ValueError as exc:
    print("refused:", str(exc)[:70])

## 8. A generated project
Thousands of small declarations, a few of them flagged. Counting the
flags points straight at the lines to review, and the partition check
proves no line was lost or read twice.



In [ ]:
lines = ["namespace Big"]
for i in range(5_000):
    proof = "by sorry" if i % 1_000 == 999 else "rfl"
    lines.append(f"theorem t{i} : {i} = {i} := {proof}")
lines.append("end Big")
big = read("\n".join(lines) + "\n", "Big.lean")

flags = Counter(flag for c in big for flag in c.get("flags", ()))
print(f"{len(big)} chunks; flags: {dict(flags)}")
print([f"{c['declaration']} at line {c['line_start']}" for c in big if c.get("flags")])

covered = [n for c in big for n in range(c["line_start"], c["line_end"] + 1)]
assert covered == list(range(1, len(lines) + 1)), "lines lost or repeated"
print(f"all {len(lines):,} lines are in exactly one chunk")

.. tags::

   model-workflow: corpus
   plot-type: text
   level: intermediate
   purpose: showcase



In [ ]:
# This is the last cell